In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import wandb
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),  # normalizes to [0,1]
])

dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128)
test_loader = DataLoader(test_data, batch_size=128)

In [3]:
#Auto Encoder
class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 28*28),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon.view(-1, 1, 28, 28)

In [21]:
#Variational Autoencoder (VAE)
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fc1 = nn.Linear(28*28, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.fc2 = nn.Linear(latent_dim, 256)
        self.fc3 = nn.Linear(256, 28*28)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return self.fc3(h)    # 🔥 FORCE sigmoid here

    def forward(self, x):
        x = x.view(-1, 28*28)

        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)

        recon = self.decode(z)  # ✅ MUST use decode()

        return recon.view(-1, 1, 28, 28), mu, logvar

In [24]:
#Loss Function
def ae_loss(recon, x, loss_type="bce"):
    if loss_type == "bce":
        return nn.functional.binary_cross_entropy(recon, x, reduction='sum')
    else:
        return nn.functional.mse_loss(recon, x, reduction='sum')


def vae_loss(recon, x, mu, logvar, loss_type="bce"):
    recon = recon.view(-1, 28*28)
    x = x.view(-1, 28*28)

    if loss_type == "bce":
        recon_loss = nn.functional.binary_cross_entropy_with_logits(recon, x, reduction='sum')
    else:
        recon_loss = nn.functional.mse_loss(torch.sigmoid(recon), x, reduction='sum')

    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + kl

In [25]:
#Training 
def train(model, loader, optimizer, loss_type="bce", is_vae=False):
    model.train()
    total_loss = 0

    for i, (x, _) in enumerate(loader):
        x = x.to(device)
        optimizer.zero_grad()

        if is_vae:
            recon, mu, logvar = model(x)

            # ✅ DEBUG PRINT (only first batch)
            if i == 0:
                print("MIN:", recon.min().item(), "MAX:", recon.max().item())

            loss = vae_loss(recon, x, mu, logvar, loss_type)
        else:
            recon = model(x)

            # ✅ DEBUG PRINT
            if i == 0:
                print("MIN:", recon.min().item(), "MAX:", recon.max().item())

            loss = ae_loss(recon, x, loss_type)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader.dataset)

In [26]:
def evaluate(model, loader, loss_type="bce", is_vae=False):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)

            if is_vae:
                recon, mu, logvar = model(x)
                loss = vae_loss(recon, x, mu, logvar, loss_type)
            else:
                recon = model(x)
                loss = ae_loss(recon, x, loss_type)

            total_loss += loss.item()

    return total_loss / len(loader.dataset)

In [27]:
#Optimizer Selector
def get_optimizer(name, model):
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=0.01)
    elif name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=0.001)
    else:
        return optim.Adam(model.parameters(), lr=0.001)

In [28]:
#Training Loop + W&B Logging
def run_experiment(model_type="ae", latent_dim=8, loss_type="bce", opt_name="adam", epochs=10):
    wandb.init(project="autoencoder-vs-vae", config={
        "model": model_type,
        "latent_dim": latent_dim,
        "loss": loss_type,
        "optimizer": opt_name
    })

    if model_type == "ae":
        model = Autoencoder(latent_dim).to(device)
        is_vae = False
    else:
        model = VAE(latent_dim).to(device)
        is_vae = True

    optimizer = get_optimizer(opt_name, model)
   
    for epoch in range(epochs):
        train_loss = train(model, train_loader, optimizer, loss_type, is_vae)
        val_loss = evaluate(model, val_loader, loss_type, is_vae)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        print(f"Epoch {epoch}: Train {train_loss:.4f}, Val {val_loss:.4f}")

    return model

In [29]:
#Latent Space Interpolation
def interpolate(model, loader, is_vae=False):
    model.eval()
    x, _ = next(iter(loader))
    x = x.to(device)

    x1, x2 = x[0:1], x[1:2]

    with torch.no_grad():
        if is_vae:
            mu1, _ = model.encode(x1.view(-1, 28*28))
            mu2, _ = model.encode(x2.view(-1, 28*28))
        else:
            mu1 = model.encoder(x1)
            mu2 = model.encoder(x2)

        alphas = np.linspace(0, 1, 10)
        images = []

        for a in alphas:
            z = (1 - a) * mu1 + a * mu2

            if is_vae:
                recon = model.decode(z)
            else:
                recon = model.decoder(z)

            images.append(recon.view(28, 28).cpu().numpy())

    # plot
    fig, axes = plt.subplots(1, 10, figsize=(15, 2))
    for i, img in enumerate(images):
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
    plt.show()

In [30]:
latent_dims = [2, 8, 16, 32]
optimizers = ["sgd", "rmsprop", "adam"]
losses = ["bce", "mse"]

for ld in latent_dims:
    for opt in optimizers:
        for loss in losses:
            run_experiment("ae", ld, loss, opt)
            run_experiment("vae", ld, loss, opt)

MIN: 0.3433910012245178 MAX: 0.6553894877433777
Epoch 0: Train 21735.8323, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 1: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 2: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 3: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 4: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 5: Train 21802.6430, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 6: Train 21802.6430, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 7: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 8: Train 21802.6431, Val 21796.1668
MIN: 0.0 MAX: 1.0
Epoch 9: Train 21802.6430, Val 21796.1668


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,▁█████████
val_loss,▁▁▁▁▁▁▁▁▁▁
epoch,9
train_loss,21802.64304
val_loss,21796.16679


MIN: -1.6764676570892334 MAX: 1.960792899131775
Epoch 0: Train nan, Val nan
MIN: nan MAX: nan
Epoch 1: Train nan, Val nan
MIN: nan MAX: nan
Epoch 2: Train nan, Val nan
MIN: nan MAX: nan
Epoch 3: Train nan, Val nan
MIN: nan MAX: nan
Epoch 4: Train nan, Val nan
MIN: nan MAX: nan
Epoch 5: Train nan, Val nan
MIN: nan MAX: nan
Epoch 6: Train nan, Val nan
MIN: nan MAX: nan
Epoch 7: Train nan, Val nan
MIN: nan MAX: nan
Epoch 8: Train nan, Val nan
MIN: nan MAX: nan
Epoch 9: Train nan, Val nan


epoch,▁▂▃▃▄▅▆▆▇█
+2,...
epoch,9
train_loss,nan
val_loss,nan


MIN: 0.3737744390964508 MAX: 0.6304540038108826
Epoch 0: Train 173.9225, Val 175.8175
MIN: 0.0 MAX: 1.0
Epoch 1: Train 175.8289, Val 175.8175
MIN: 0.0 MAX: 1.0
Epoch 2: Train 175.8757, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 3: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 4: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 5: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 6: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 7: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 8: Train 175.8465, Val 175.8391
MIN: 0.0 MAX: 1.0
Epoch 9: Train 175.8465, Val 175.8391


...

MIN: -1.2181758880615234 MAX: 1.177457571029663
Epoch 0: Train 301.3361, Val 271.0153
MIN: -16.506746292114258 MAX: 3.1394965648651123
Epoch 1: Train 265.2518, Val 260.3644
MIN: -17.95234489440918 MAX: 4.217447280883789
Epoch 2: Train 257.1724, Val 254.1911
MIN: -19.565086364746094 MAX: 4.1964592933654785
Epoch 3: Train 252.6369, Val 250.6808
MIN: -27.827842712402344 MAX: 4.053646564483643
Epoch 4: Train 249.9363, Val 248.6424
MIN: -27.855697631835938 MAX: 4.221951961517334
Epoch 5: Train 248.0651, Val 247.3006
MIN: -26.52837371826172 MAX: 4.357036590576172
Epoch 6: Train 246.8999, Val 246.8263
MIN: -31.485652923583984 MAX: 4.443052768707275
Epoch 7: Train 245.9895, Val 245.4806
MIN: -35.312110900878906 MAX: 3.9631099700927734
Epoch 8: Train 245.2955, Val 244.7619
MIN: -33.84081268310547 MAX: 3.772111654281616
Epoch 9: Train 244.6992, Val 244.4179


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▁▁▁▁▁
val_loss,█▅▄▃▂▂▂▁▁▁
epoch,9
train_loss,244.69915
val_loss,244.41787


MIN: 0.4405980408191681 MAX: 0.5563552975654602
Epoch 0: Train 27.1386, Val 15.4001
MIN: 2.02172480145979e-11 MAX: 0.9830945134162903
Epoch 1: Train 13.7887, Val 12.4822
MIN: 4.065758688921728e-13 MAX: 0.9871713519096375
Epoch 2: Train 11.6671, Val 11.0468
MIN: 3.7551213495027463e-14 MAX: 0.9879085421562195
Epoch 3: Train 10.5770, Val 10.2283
MIN: 4.280619488836483e-15 MAX: 0.9884583353996277
Epoch 4: Train 9.8350, Val 9.6227
MIN: 2.4400552226937515e-16 MAX: 0.9916463494300842
Epoch 5: Train 9.2625, Val 9.1413
MIN: 1.7218157103001883e-14 MAX: 0.9921221137046814
Epoch 6: Train 8.8530, Val 8.8121
MIN: 2.585601688113846e-16 MAX: 0.9912303686141968
Epoch 7: Train 8.5112, Val 8.5064
MIN: 8.61846296187247e-17 MAX: 0.9964520931243896
Epoch 8: Train 8.2377, Val 8.2749
MIN: 1.0833847009881093e-14 MAX: 0.9871547222137451
Epoch 9: Train 7.9964, Val 8.1272


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▃▂▂▂▁▁▁▁▁
val_loss,█▅▄▃▂▂▂▁▁▁
epoch,9
train_loss,7.99637
val_loss,8.1272


MIN: -1.3002322912216187 MAX: 1.1877654790878296
Epoch 0: Train 44.6664, Val 33.9958
MIN: -9.634929656982422 MAX: 3.305253505706787
Epoch 1: Train 31.9344, Val 30.3129
MIN: -10.58049488067627 MAX: 3.4548611640930176
Epoch 2: Train 29.3359, Val 28.5267
MIN: -13.619731903076172 MAX: 3.951906442642212
Epoch 3: Train 28.0346, Val 27.6991
MIN: -18.325439453125 MAX: 4.422409534454346
Epoch 4: Train 27.2515, Val 27.0520
MIN: -18.780101776123047 MAX: 4.100244998931885
Epoch 5: Train 26.8435, Val 26.6532
MIN: -20.659461975097656 MAX: 4.140265464782715
Epoch 6: Train 26.4767, Val 26.3977
MIN: -21.097557067871094 MAX: 3.549940347671509
Epoch 7: Train 26.2059, Val 26.0911
MIN: -27.543014526367188 MAX: 4.1539306640625
Epoch 8: Train 26.0300, Val 26.0626
MIN: -26.522851943969727 MAX: 3.6644880771636963
Epoch 9: Train 25.8204, Val 25.7804
